# Chapter 25 — Five Approaches to Context Management
## Demo Notebook

**Core claim:** The locus of control for context curation determines the failure mode of a long-running agent — not the size of the context window.

**What this notebook demonstrates:**
1. Passive agent accumulating dead-end vocabulary until it produces the wrong answer
2. ACM agent with proactive removal — same task, clean context, correct answer
3. Measurable comparison: vocabulary drift per turn, final answer correctness

**Stop at Cell 8** (Human Decision Node) and document your decision before proceeding.

In [10]:
# Setup
from dataclasses import dataclass, field
from typing import Optional

TASK = """Debug a function that fails approximately 1 time in 20 under
concurrent load. The function handles session token refresh. Logs show
the failure never reproduces in single-threaded test environments."""

DEAD_END_A_VOCAB = ["pool_exhaustion", "connection_pool", "max_connections",
                    "pool_timeout", "db_pool", "acquire_connection"]
DEAD_END_B_VOCAB = ["cache_invalidation", "cache_miss", "stale_cache",
                    "invalidate_cache", "cache_key", "cache_ttl"]

def measure_vocab_contamination(messages, vocab):
    full_text = " ".join(
        m.get("content", "") if isinstance(m, dict) else str(m)
        for m in messages
    )
    return sum(full_text.lower().count(term.lower()) for term in vocab)

print(f"Task: {TASK[:80]}...")
print(f"Dead-end A: {DEAD_END_A_VOCAB}")
print(f"Dead-end B: {DEAD_END_B_VOCAB}")

Task: Debug a function that fails approximately 1 time in 20 under
concurrent load. Th...
Dead-end A: ['pool_exhaustion', 'connection_pool', 'max_connections', 'pool_timeout', 'db_pool', 'acquire_connection']
Dead-end B: ['cache_invalidation', 'cache_miss', 'stale_cache', 'invalidate_cache', 'cache_key', 'cache_ttl']


## The Stub Tool Environment

Simulates tool calls against a token-refresh codebase.

**Injected misleading outputs (identical for both agents):**
- **Turn 5:** False positive — connection pool exhaustion
- **Turn 12:** False positive — cache invalidation failure

The real bug (clock skew) only surfaces if the agent maintains the constraint *'never reproduces single-threaded'*.

In [11]:
class StubToolEnvironment:
    def __init__(self):
        self.turn = 0

    def call_tool(self, tool_name, args):
        self.turn += 1
        return self._get_result(tool_name, args)

    def _get_result(self, tool_name, args):
        if self.turn == 5:
            return ("WARNING: connection_pool max_connections=10 reached. "
                    "pool_exhaustion detected during load test. "
                    "db_pool.acquire_connection() timing out. pool_timeout in token refresh path.")
        if self.turn == 12:
            return ("cache_invalidation event on token_refresh key. "
                    "stale_cache entries: 3 expired tokens not invalidated. "
                    "cache_miss rate 34% under load. cache_ttl mismatch in refresh handler.")
        if self.turn > 18 and "clock" in str(args).lower():
            return ("CONFIRMED: token_issued_at uses server local time. "
                    "NTP sync drift causes token_expiry check to fail ~1/20 requests under load. "
                    "Fix: UTC timestamps + monotonic clock. "
                    "Explains staging immunity: single-threaded, no NTP pressure.")
        return f"Turn {self.turn}: inconclusive. No consistent failure in isolation."

print("StubToolEnvironment ready.")

StubToolEnvironment ready.


## Approach 1: Passive Agent Loop

Exact structure from Section 3. The only decision the loop makes about context content is **positional**: `messages.pop(0)` drops the oldest.

Watch vocabulary contamination accumulate after turns 5 and 12.

In [12]:
def simulate_agent_turn(messages, env, agent_type="passive"):
    dea = measure_vocab_contamination(messages, DEAD_END_A_VOCAB)
    deb = measure_vocab_contamination(messages, DEAD_END_B_VOCAB)
    turn = env.turn + 1

    if turn <= 4:
        result = env.call_tool("read_file", {"path": "auth.py"})
        return f"[{turn}] Reading middleware. Candidates: clock skew, concurrency.", result
    elif turn == 5:
        result = env.call_tool("run_test", {"hypothesis": "concurrency"})
        return f"[{turn}] Concurrency test: {result[:80]}...", result
    elif 6 <= turn <= 11:
        if dea > 3 and agent_type == "passive":
            r = f"[{turn}] Investigating connection_pool. pool_exhaustion present {dea}x. Testing db_pool timeout."
        else:
            r = f"[{turn}] Constraint active: never single-threaded. Investigating load timing."
        result = env.call_tool("search_logs", {"pattern": "pool" if dea > 3 else "timing"})
        return r, result
    elif turn == 12:
        result = env.call_tool("check_timing", {"focus": "cache"})
        return f"[{turn}] Cache invalidation signal: {result[:80]}...", result
    elif 13 <= turn <= 18:
        total = dea + deb
        if total > 8 and agent_type == "passive":
            r = f"[{turn}] CONTAMINATED (pool={dea}, cache={deb}). Pursuing combined pool+cache hypothesis."
            result = env.call_tool("run_test", {"hypothesis": "pool_cache"})
        else:
            r = f"[{turn}] Constraint: load-only, never staging. Clock timing focus."
            result = env.call_tool("check_timing", {"focus": "clock"})
        return r, result
    else:
        if agent_type == "passive" and dea + deb > 10:
            return (f"[{turn}] CONCLUSION: Root cause = pool_exhaustion + cache_invalidation. "
                    "Fix: increase max_connections, align cache_ttl."), None
        result = env.call_tool("check_timing", {"hypothesis": "clock_skew_utc"})
        return (f"[{turn}] Clock skew CONFIRMED. token_issued_at local time. "
                "NTP drift ~1/20 under load. Fix: UTC + monotonic clock. Staging immune."), result


def run_passive_agent(task, max_turns=22):
    messages = [{"role": "user", "content": task}]
    env = StubToolEnvironment()
    vocab_drift, session_log = [], []

    for t in range(max_turns):
        reasoning, tool_result = simulate_agent_turn(messages, env, "passive")
        messages.append({"role": "assistant", "content": reasoning})
        contamination = measure_vocab_contamination(messages, DEAD_END_A_VOCAB + DEAD_END_B_VOCAB)
        vocab_drift.append(contamination)
        session_log.append({"turn": t+1, "vocab": contamination, "snippet": reasoning[:100]})
        if tool_result:
            messages.append({"role": "user", "content": tool_result})
        while len(messages) > 40:  # passive truncation
            messages.pop(0)
        if "CONCLUSION" in reasoning or "CONFIRMED" in reasoning:
            break

    final = messages[-2]["content"] if len(messages) > 1 else "No answer"
    return {"final_answer": final, "turns": t+1, "context_size": len(messages),
            "vocab_drift": vocab_drift, "session_log": session_log,
            "correct": "clock" in final.lower() or "UTC" in final}

print("Passive agent defined.")

Passive agent defined.


In [13]:
passive_result = run_passive_agent(TASK)

print("=" * 60)
print("PASSIVE AGENT RESULTS")
print("=" * 60)
print(f"\nFinal Answer:\n{passive_result['final_answer']}")
print(f"\nCorrect: {passive_result['correct']}")
print(f"Turns: {passive_result['turns']} | Context: {passive_result['context_size']} messages")
print(f"\nVocabulary contamination (dead-end tokens in context view):")
for e in passive_result['session_log']:
    bar = chr(9608) * e['vocab']
    m = " ← DEAD-END A INJECTED" if e['turn'] == 5 else ("← DEAD-END B INJECTED" if e['turn'] == 12 else "")
    print(f"  Turn {e['turn']:2d}: {bar:25s} ({e['vocab']:2d}) {m}")

PASSIVE AGENT RESULTS

Final Answer:
Turn 18: inconclusive. No consistent failure in isolation.

Correct: False
Turns: 19 | Context: 38 messages

Vocabulary contamination (dead-end tokens in context view):
  Turn  1:                           ( 0) 
  Turn  2:                           ( 0) 
  Turn  3:                           ( 0) 
  Turn  4:                           ( 0) 
  Turn  5: ███                       ( 3)  ← DEAD-END A INJECTED
  Turn  6: ████████████              (12) 
  Turn  7: ███████████████           (15) 
  Turn  8: ██████████████████        (18) 
  Turn  9: █████████████████████     (21) 
  Turn 10: ████████████████████████  (24) 
  Turn 11: ███████████████████████████ (27) 
  Turn 12: █████████████████████████████ (29) ← DEAD-END B INJECTED
  Turn 13: █████████████████████████████████ (33) 
  Turn 14: █████████████████████████████████ (33) 
  Turn 15: █████████████████████████████████ (33) 
  Turn 16: █████████████████████████████████ (33) 
  Turn 17: ██████████████

In [14]:
# ============================================================
# MANDATORY HUMAN DECISION NODE
# ============================================================
# The ACM agent uses PROACTIVE remove_context calls:
# dead-ends removed AT THE MOMENT OF INVALIDATION.
#
# I REJECTED reactive ACM (remove only when crowded) because
# it masks the architectural insight: leverage comes from
# TIMING of removal, not just having the tool.
#
# Proactive: attention sink never forms
# Reactive:  sink forms, then gets cleaned — partial benefit
#
# DECISION REQUIRED before proceeding:
# Does your deployment use proactive or reactive removal?
# ============================================================

YOUR_DECISION = "proactive"  # Can be anything.  "proactive" or "reactive" + reasoning

if YOUR_DECISION is None:
    raise NotImplementedError(
        "\n\nMANDATORY HUMAN DECISION NODE\n"
        "Set YOUR_DECISION = 'proactive' or 'reactive' with your reasoning.\n"
        "Comment out this raise, then re-run.\n"
    )

print(f"Decision documented: {YOUR_DECISION}")
print("Proceeding to ACM implementation.")

Decision documented: proactive
Proceeding to ACM implementation.


In [15]:
@dataclass
class Msg:
    id: str; role: str; content: str
    pinned: bool = False; removed: bool = False
    summary: Optional[str] = None; label: Optional[str] = None

@dataclass
class ACMAgent:
    log: list = field(default_factory=list)
    directives: list = field(default_factory=list)
    n: int = 0
    tool_log: list = field(default_factory=list)

    def _id(self):
        self.n += 1; return f"msg-{self.n:03d}"

    def append(self, role, content):
        mid = self._id()
        self.log.append(Msg(id=mid, role=role, content=content))
        return mid

    def get_view(self):
        view = {m.id: Msg(id=m.id, role=m.role, content=m.content,
                          pinned=m.pinned, removed=m.removed,
                          summary=m.summary, label=m.label) for m in self.log}
        for d in self.directives:
            if d["type"] == "pin":
                for mid in d["ids"]:
                    if mid in view: view[mid].pinned = True
            elif d["type"] == "remove":
                for mid in d["ids"]:
                    if mid in view and not view[mid].pinned:
                        view[mid].removed = True
                        view[mid].summary = d.get("summary")
                        view[mid].label = d.get("label")
            elif d["type"] == "retrieve":
                for m in view.values():
                    if m.label == d.get("label"): m.removed = False
        result, seen = [], set()
        for m in self.log:
            v = view[m.id]
            if not v.removed:
                result.append({"role": v.role, "content": v.content, "id": v.id})
            elif v.summary and v.label and v.label not in seen:
                result.append({"role": "system", "content": f"[SUMMARY:{v.label}] {v.summary}", "id": v.id})
                seen.add(v.label)
        return result

    def remove_context(self, ids, summary=None, label=None):
        self.directives.append({"type": "remove", "ids": ids, "summary": summary, "label": label})
        self.tool_log.append({"tool": "remove_context", "ids": ids, "label": label})

    def pin(self, ids):
        self.directives.append({"type": "pin", "ids": ids})
        self.tool_log.append({"tool": "pin", "ids": ids})

    def retrieve_context(self, label=None):
        self.directives.append({"type": "retrieve", "label": label})
        self.tool_log.append({"tool": "retrieve_context", "label": label})


def run_acm_agent(task, max_turns=22):
    agent = ACMAgent()
    env = StubToolEnvironment()
    vocab_drift, session_log = [], []
    task_id = agent.append("user", task)
    agent.pin([task_id])  # PROACTIVE: pin before any work
    removed_a = removed_b = False

    for t in range(max_turns):
        view = agent.get_view()
        reasoning, tool_result = simulate_agent_turn(view, env, "acm")
        agent.append("assistant", reasoning)

        # PROACTIVE: remove dead-end A at moment of invalidation
        if t == 5 and not removed_a:
            tr_id = agent.append("user", tool_result or "")
            agent.remove_context(
                ids=[tr_id],
                summary=("Dead-end A INVALIDATED: pool_exhaustion false positive. "
                         "'Never single-threaded' constraint rules out connection limits. Cleared."),
                label="dead-end-A")
            removed_a = True; tool_result = None

        # PROACTIVE: remove dead-end B at moment of invalidation
        if t == 11 and not removed_b:
            tr_id = agent.append("user", tool_result or "")
            agent.remove_context(
                ids=[tr_id],
                summary=("Dead-end B INVALIDATED: cache_invalidation false positive. "
                         "Staging immunity rules out cache-layer bugs. Cleared."),
                label="dead-end-B")
            removed_b = True; tool_result = None

        if tool_result: agent.append("user", tool_result)

        view_text = " ".join(m["content"] for m in agent.get_view())
        contamination = sum(view_text.lower().count(term.lower())
                           for term in DEAD_END_A_VOCAB + DEAD_END_B_VOCAB)
        vocab_drift.append(contamination)
        session_log.append({"turn": t+1, "vocab": contamination,
                            "view": len(agent.get_view()), "log": len(agent.log)})
        if "CONCLUSION" in reasoning or "CONFIRMED" in reasoning: break

    cv = agent.get_view()
    final = cv[-2]["content"] if len(cv) > 1 else "No answer"
    return {"final_answer": final, "turns": t+1,
            "context_view_size": len(cv), "log_size": len(agent.log),
            "vocab_drift": vocab_drift, "session_log": session_log,
            "tool_log": agent.tool_log,
            "correct": "clock" in final.lower() or "UTC" in final}

print("ACM agent defined.")

ACM agent defined.


In [16]:
acm_result = run_acm_agent(TASK)

print("=" * 60)
print("ACM AGENT RESULTS")
print("=" * 60)
print(f"\nFinal Answer:\n{acm_result['final_answer']}")
print(f"\nCorrect: {acm_result['correct']}")
print(f"Turns: {acm_result['turns']} | View: {acm_result['context_view_size']} | Log: {acm_result['log_size']}")
print(f"\nACM Tool Call Log (auditable record):")
for c in acm_result['tool_log']:
    print(f"  {c['tool']}({c.get('label') or c.get('ids', [])})")
print(f"\nVocabulary contamination in CONTEXT VIEW:")
for e in acm_result['session_log']:
    bar = chr(9608) * e['vocab']
    m = " ← remove_context(dead-end-A)" if e['turn'] == 6 else (
        " ← remove_context(dead-end-B)" if e['turn'] == 12 else "")
    print(f"  Turn {e['turn']:2d}: {bar:25s} ({e['vocab']:2d}) {m}")

ACM AGENT RESULTS

Final Answer:
[19] Clock skew CONFIRMED. token_issued_at local time. NTP drift ~1/20 under load. Fix: UTC + monotonic clock. Staging immune.

Correct: True
Turns: 19 | View: 39 | Log: 39

ACM Tool Call Log (auditable record):
  pin(['msg-001'])
  remove_context(dead-end-A)
  remove_context(dead-end-B)

Vocabulary contamination in CONTEXT VIEW:
  Turn  1:                           ( 0) 
  Turn  2:                           ( 0) 
  Turn  3:                           ( 0) 
  Turn  4:                           ( 0) 
  Turn  5: █████████                 ( 9) 
  Turn  6: ██████████                (10)  ← remove_context(dead-end-A)
  Turn  7: ██████████                (10) 
  Turn  8: ██████████                (10) 
  Turn  9: ██████████                (10) 
  Turn 10: ██████████                (10) 
  Turn 11: ██████████                (10) 
  Turn 12: █████████████             (13)  ← remove_context(dead-end-B)
  Turn 13: █████████████             (13) 
  Turn 14: ███████

In [17]:
print("=" * 60)
print("SIDE-BY-SIDE COMPARISON")
print("=" * 60)

p = passive_result['vocab_drift']
a = acm_result['vocab_drift']

print(f"\n{'Metric':<42} {'Passive':>8} {'ACM':>8}")
print("-" * 60)
print(f"{'Correct final answer':<42} {'YES' if passive_result['correct'] else 'NO':>8} {'YES' if acm_result['correct'] else 'NO':>8}")
print(f"{'Peak dead-end vocabulary count':<42} {max(p):>8} {max(a):>8}")
print(f"{'Final dead-end vocabulary count':<42} {p[-1]:>8} {a[-1]:>8}")
print(f"{'Final context size (messages)':<42} {passive_result['context_size']:>8} {acm_result['context_view_size']:>8}")

print(f"\n{'Turn':<6} {'Passive':>10} {'ACM':>8} {'Diff':>6}")
print("-" * 34)
for i in range(min(len(p), len(a))):
    diff = p[i] - a[i]
    mark = " ◄ DIVERGENCE BEGINS" if i == 5 and diff > 0 else ""
    print(f"  {i+1:<4} {p[i]:>10} {a[i]:>8} {diff:>6}{mark}")

print("\n--- Final Answers ---")
print(f"Passive: {passive_result['final_answer'][:200]}")
print(f"\nACM:     {acm_result['final_answer'][:200]}")

print("\n--- The Mechanism ---")
print(f"Turn 6: passive context = {p[5] if len(p)>5 else 0} dead-end tokens | ACM view = {a[5] if len(a)>5 else 0}")
print("Passive: attention sink from pool_exhaustion vocab biased turns 6-18 toward wrong hypothesis")
print("ACM: remove_context at turn 6 replaced 300+ tokens with 2-line tombstone")
print("     Attention sink never formed. Clock-skew analysis ran on clean context.")

SIDE-BY-SIDE COMPARISON

Metric                                      Passive      ACM
------------------------------------------------------------
Correct final answer                             NO      YES
Peak dead-end vocabulary count                   37       13
Final dead-end vocabulary count                  37       13
Final context size (messages)                    38       39

Turn      Passive      ACM   Diff
----------------------------------
  1             0        0      0
  2             0        0      0
  3             0        0      0
  4             0        0      0
  5             3        9     -6
  6            12       10      2 ◄ DIVERGENCE BEGINS
  7            15       10      5
  8            18       10      8
  9            21       10     11
  10           24       10     14
  11           27       10     17
  12           29       13     16
  13           33       13     20
  14           33       13     20
  15           33       13     20
  16     

## The Exercise

**Before checking the comparison output above**, answer this in writing:

> At the divergence turn, what is different about the two agents' Context Views, and why does that difference produce a different output? Name the specific tokens present in the passive agent's context and absent from the ACM agent's context. Trace the attention sink mechanism from Section 2 to the output divergence you observe.

**Grading criterion:**

❌ *"The ACM agent did better because it managed context."*

✅ *"At turn 6, the passive agent's context contained N instances of `pool_exhaustion` from the turn-5 injection, creating an attention sink that biased hypothesis generation toward connection management vocabulary. The ACM agent's context view contained a 2-line tombstone with no high-salience vocabulary residue — the attention sink never formed because remove_context fired at the moment of invalidation, not reactively."*

**Your answer:**

[Write here]

In [18]:
# ============================================================
# FAILURE MODE DEMONSTRATION: ACM METACOGNITIVE ERROR
# The failure is AUDITABLE — contrast with passive's silent failure.
# ============================================================

print("=" * 60)
print("ACM FAILURE MODE: METACOGNITIVE ERROR")
print("=" * 60)

agent_err = ACMAgent()
task_id = agent_err.append("user", TASK)
# NO pin() — this is the metacognitive error
agent_err.append("assistant", "[Turn 1] Reading middleware. Constraint noted.")
agent_err.append("assistant", "[Turn 2] Constraints embedded in reasoning. Task message is redundant.")

# THE ERROR: remove without pinning first
agent_err.remove_context(
    ids=[task_id],
    summary="Task: debug intermittent token refresh failure.",
    label="task-removed"
)

print(f"\nERROR COMMITTED:")
print(f"  remove_context('{task_id}') called without prior pin()")
print(f"  LOST from context view: 'never reproduces single-threaded'")
print(f"  This constraint rules out entire classes of deterministic bugs")

print(f"\nBUT — the failure is AUDITABLE:")
print(f"  Tool call log: remove_context on {task_id} at turn 2")
print(f"  No prior pin() call visible in log — error is immediately identifiable")
print(f"  Original in conversation log — nothing destroyed")

print(f"\nRECOVERY:")
agent_err.retrieve_context(label="task-removed")
agent_err.pin([task_id])
print(f"  retrieve_context('task-removed') → restored")
print(f"  pin('{task_id}') → protected for remainder of session")

print(f"\nCONTRAST — passive truncation failure:")
print("  No discrete event. No tool call log. No recovery point.")
print("  Contamination is continuous, distributed, structurally inevitable.")
print("  Forensic analyst cannot find WHEN failure began — there was no when.")
print()
print("ACM tool call log (full audit trail):")
for c in agent_err.tool_log:
    print(f"  {c}")

ACM FAILURE MODE: METACOGNITIVE ERROR

ERROR COMMITTED:
  remove_context('msg-001') called without prior pin()
  LOST from context view: 'never reproduces single-threaded'
  This constraint rules out entire classes of deterministic bugs

BUT — the failure is AUDITABLE:
  Tool call log: remove_context on msg-001 at turn 2
  No prior pin() call visible in log — error is immediately identifiable
  Original in conversation log — nothing destroyed

RECOVERY:
  retrieve_context('task-removed') → restored
  pin('msg-001') → protected for remainder of session

CONTRAST — passive truncation failure:
  No discrete event. No tool call log. No recovery point.
  Contamination is continuous, distributed, structurally inevitable.
  Forensic analyst cannot find WHEN failure began — there was no when.

ACM tool call log (full audit trail):
  {'tool': 'remove_context', 'ids': ['msg-001'], 'label': 'task-removed'}
  {'tool': 'retrieve_context', 'label': 'task-removed'}
  {'tool': 'pin', 'ids': ['msg-001'